In [1]:
from pathlib import Path

# Find repo root
REPO_ROOT = Path.cwd().parent
print(f"Repo root: {REPO_ROOT}")

REPORT_ROOT = REPO_ROOT / "report"

FIGSIZE = (20,18)
DPI = 100
GENERATE_PLOTS = False

Repo root: /Users/jedrek/Documents/Studium Volkswirschaftslehre/4. Semester/DEDA Project/DEDA_LLM_Spatial_Hotelling


In [2]:
import pandas as pd
import geopandas as gpd
import numpy as np
from pathlib import Path
import sys
import json
from shapely.geometry import shape
from hotelling.spatial.admin import join_lor_names

# Add src to path for imports
sys.path.insert(0, str(Path.cwd().parent / 'src'))

from hotelling.spatial.boundaries import load_boundary

PATH_RAW = REPO_ROOT / Path('data/raw')
PATH_PROCESSED = REPO_ROOT / Path('data/processed')

# Midpoint table (center coordinates)
zensus = gpd.read_parquet(PATH_RAW / 'zensus2022_grid.parquet')
zensus_filtered = gpd.read_parquet(PATH_RAW / 'zensus2022_grid_filtered.parquet')
lor = gpd.read_parquet(PATH_PROCESSED / 'lor.parquet')

# CRITICAL FIX: berlin.geojson has EPSG:3035 coordinates but geopandas auto-detects as EPSG:4326
# We must force the correct CRS instead of transforming from the wrong one
with open(PATH_RAW / 'city_boundary_Berlin.geojson', 'r') as f:
    berlin_json = json.load(f)
berlin = gpd.GeoDataFrame([1], geometry=[shape(berlin_json['geometry'])], crs='EPSG:3035')

boundary = load_boundary(PATH_RAW / 'relation_boundary_14983.geojson')

In [3]:
from hotelling.spatial.census import build_grid_polygons

grid = gpd.read_parquet(PATH_PROCESSED / 'pop_grid.parquet')

# Pop grid was saved with point geometry (midpoints). Convert to 100m square polygons.
grid = build_grid_polygons(grid)
grid['index'] = grid.index
print(f"Grid: {len(grid)} cells as square polygons")


Grid: 16170 cells as square polygons


In [4]:
from hotelling.spatial.city_data import process_ihk_data

grid = process_ihk_data(grid, PATH_RAW / '2023_12_IHK_Berlin_Gewerbedaten.csv')
print(f"Grid with IHK employment: empl column added, total empl={grid['empl'].sum():.0f}")


Grid with IHK employment: empl column added, total empl=1180916


In [5]:
from hotelling.spatial.city_data import process_gebaeude_stadtstruktur

gebaeude_stadtstruktur = process_gebaeude_stadtstruktur(
    gebaeude_path=PATH_RAW / 'gebaeude.gpkg',
    stadtstruktur_path=PATH_RAW / 'stadtstruktur.gpkg',
    ihk_path=PATH_RAW / '2023_12_IHK_Berlin_Gewerbedaten.csv',
)
print(f"gebaeude_stadtstruktur: {len(gebaeude_stadtstruktur)} buildings")
print(f"  approx_empl total: {gebaeude_stadtstruktur['approx_empl'].sum():.0f}")
print(f"  buildings with empl>0: {(gebaeude_stadtstruktur['approx_empl'] > 0).sum()}")


gebaeude_stadtstruktur: 543611 buildings
  approx_empl total: 1135606
  buildings with empl>0: 41201


In [6]:
df_to_plot = gebaeude_stadtstruktur[gebaeude_stadtstruktur['aog'].isna()]

# Plot with berlin boundary
import matplotlib.pyplot as plt
if GENERATE_PLOTS:
    fig, ax = plt.subplots(figsize=FIGSIZE, dpi=DPI)
    berlin.to_crs('EPSG:25833').plot(ax=ax, color='none', edgecolor='black', linewidth=1)
    df_to_plot.plot(ax=ax, color='red', markersize=10)
    plt.title('Buildings with no stadtstruktur match (red) and Berlin boundary')
    plt.show()


In [7]:
df_to_plot = gebaeude_stadtstruktur[gebaeude_stadtstruktur['aog'].isna()]

# Plot with berlin boundary
if GENERATE_PLOTS:
    fig, ax = plt.subplots(figsize=FIGSIZE, dpi=DPI)
    berlin.to_crs('EPSG:25833').plot(ax=ax, color='none', edgecolor='black', linewidth=1)
    df_to_plot.plot(ax=ax, color='red')
    plt.title('Buildings with no stadtstruktur match (red) and Berlin boundary')
    plt.show()


In [8]:
# IHK→building distance matching is performed inside process_gebaeude_stadtstruktur().
# Inspect gebaeude_stadtstruktur['empl'] and ['approx_empl'] for capped employment totals.
gebaeude_stadtstruktur[['empl', 'approx_empl', 'employee_hard_cap']].describe()


/Users/jedrek/miniforge3/envs/py314/lib/python3.14/site-packages/numpy/lib/_function_base_impl.py:4671: RuntimeWarning: invalid value encountered in subtract
  diff_b_a = subtract(b, a)


,empl,approx_empl,employee_hard_cap
count,543611.000000,543611.000000,5.436110e+05
mean,3.299787,2.089004,inf
std,86.878518,40.552573,NaN
min,0.000000,0.000000,1.125000e-02
25%,0.000000,0.000000,NaN
50%,0.000000,0.000000,NaN
75%,0.000000,0.000000,NaN
max,12905.000000,10000.000000,inf


In [9]:
import matplotlib.pyplot as plt

gebaeude_centroid = gebaeude_stadtstruktur.copy()
gebaeude_centroid['geometry'] = gebaeude_centroid.geometry.centroid

# Handle NaN values and normalization safely
max_empl = gebaeude_centroid['approx_empl'].replace([np.inf, -np.inf], np.nan).max()
gebaeude_centroid['approx_empl_norm'] = gebaeude_centroid['approx_empl'] / max_empl if max_empl > 0 else 0
gebaeude_centroid['approx_empl_norm'] = gebaeude_centroid['approx_empl_norm'].fillna(0)

if GENERATE_PLOTS:
    # Plot buildings colored by approx_empl with alpha dependent on approx_empl_norm
    fig, ax = plt.subplots(figsize=FIGSIZE, dpi=DPI)

    # Filter to non-zero employment buildings
    to_plot = gebaeude_centroid[gebaeude_centroid['approx_empl_norm'] != 0].copy()

    # Extract coordinates
    x = to_plot.geometry.x.values
    y = to_plot.geometry.y.values
    c = to_plot['approx_empl_norm'].values

    # Make alpha dependent on approx_empl_norm (scale to [0.1, 0.9] for visibility)
    v_min = np.vectorize(lambda x, y: min(x, y))
    alphas = v_min(0.05 + 0.1 * np.exp(c), 1)

    # Plot with variable alpha
    cmap = plt.get_cmap('viridis')
    scatter = ax.scatter(x, y, c=c, cmap=cmap, s=5, alpha=alphas, vmin=0, vmax=1)
    plt.colorbar(scatter, ax=ax, label='Normalized Employment')

    berlin.to_crs(gebaeude_stadtstruktur.crs).plot(ax=ax, color='none', edgecolor='black', linewidth=1)

    plt.title('Buildings colored and glowing by approximate employment')
    plt.show()


In [10]:
from hotelling.spatial.city_data import run_prime_location_clustering

prime_location_clusters = run_prime_location_clustering(
    gebaeude_stadtstruktur,
    k_percentile=99.5,
    min_empl=10,
    radius_m=500,
)
print(f"Prime-location clusters: {len(prime_location_clusters)}")
prime_location_clusters


Please note that the package 'aabpl' is under active development. Your currently using version 0.1.27
  11,483 buildings with approx_empl > 10
  Running AABPL (r=500 m, k=99.5th pct) over 11,483 building centroids...
Create grid with 204x252=51408
Aggregate Data from 11483 points into 205x253=51865 cells.
Points assigned to grid cell:11483/11483
sum in grid: [1027106.75545274] sum in pts [1027106.75545274]
Aggregate Data from 100000 points into 205x253=51865 cells.
Checking relation of two DataFrame with columns that are not shared: {'cell_reg', 'region_comb_nr'}. Shared columns:{'proj_y', 'id_y', 'proj_x', 'id_x'}
Threshold value for 99.5th-percentile is 15414.186417895244 for approx_empl.
Aggregate Data from 11483 points into 205x253=51865 cells.
  3 cluster(s) found:
 cluster_id           sum  n_cells  centroid_x  centroid_y
          1 186730.135735      308   13.388189   52.516853
          2  80802.011543      103   13.329965   52.503743
          3  23428.728383       30   13.44

,centroid_x,centroid_y,cluster_id,sum,n_cells,area,geometry
0,13.388189,52.516853,1,186730.135735,308,8.555556e+06,"POLYGON ((13.36397 52.501, 13.36151 52.50096, ..."
1,13.329965,52.503743,2,80802.011543,103,2.861111e+06,"POLYGON ((13.31211 52.50326, 13.31206 52.50476..."
2,13.444865,52.504734,3,23428.728383,30,8.333333e+05,"POLYGON ((13.43784 52.50349, 13.43779 52.50499..."


In [11]:
import matplotlib.pyplot as plt
import contextily as ctx

if GENERATE_PLOTS:
    fix, ax = plt.subplots(figsize=FIGSIZE, dpi=DPI)

    # Add OSM basemap
    result_web = prime_location_clusters.to_crs(berlin.crs)
    # Plot clusters and boundary
    result_web.plot(ax=ax, column='cluster_id', markersize=5, legend=True, alpha=1)
    berlin.plot(ax=ax, color='none', edgecolor='black', linewidth=1.5)
    ctx.add_basemap(ax, crs=result_web.crs, source=ctx.providers.OpenStreetMap.Mapnik, zoom=11, zorder=0, alpha = 0.3)

    plt.title('Detected employment clusters with OSM background')
    plt.show()


In [12]:
# Outputs are saved automatically by the package functions:
# data/processed/gebaeude_stadtstruktur.parquet  ← process_gebaeude_stadtstruktur()
# data/processed/prime_location_clusters.parquet ← run_prime_location_clustering()
print("Outputs saved by package functions.")


Outputs saved by package functions.


In [13]:
if not GENERATE_PLOTS:
    import nbformat, pathlib

    _nb_path = pathlib.Path(__file__) if "__file__" in dir() else None
    # Fallback: set explicitly if auto-detection unavailable
    _nb_path = pathlib.Path("GEO_02_city_data.ipynb")  # ← set once per notebook

    _nb = nbformat.read(_nb_path, as_version=4)
    for _cell in _nb.cells:
        _cell["outputs"] = []
        _cell["execution_count"] = None
    nbformat.write(_nb, _nb_path)
    print(f"Outputs cleared: {_nb_path.name}")

Outputs cleared: GEO_02_city_data.ipynb
